# Partitioning Strategies for Observability Data

The table-model examples use partitioned tables with daily partitions. Are there other partitioning strategies besides daily partitioning?

First, let's look at hourly partitioning:

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [2]:
lab.execute(r"""
CREATE TABLE `otel_logs_for_hour_partition` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200)  NULL,
  `trace_id` varchar(200)  NULL,
  `span_id` text  NULL,
  `severity_number` int  NULL,
  `severity_text` text  NULL,
  `body` text  NULL,
  `resource_attributes` variant  NULL,
  `log_attributes` variant  NULL,
  `scope_name` text  NULL,
  `scope_version` text  NULL
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)()
DISTRIBUTED BY HASH(`service_name`) BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "HOUR",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "240", -- hourly partitions for 10 days
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"enable_single_replica_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Hourly partition table')


0

Use the following command to inspect the partitions. The result contains:

```text
242 rows in set (0.01 sec)
```

This indicates that the table created partitions for the previous 10 days, plus the current partition and one pre-created partition, for a total of 242 partitions.

However, this number of partitions is excessive and creates significant metadata management pressure:


In [3]:
lab.sql(r"""
SHOW PARTITIONS FROM otel_logs_for_hour_partition;
""", title='Hourly partitions')


PartitionId,PartitionName,VisibleVersion,VisibleVersionTime,State,PartitionKey,Range,DistributionKey,Buckets,ReplicationNum,StorageMedium,CooldownTime,RemoteStoragePolicy,LastConsistencyCheckTime,DataSize,IsInMemory,ReplicaAllocation,IsMutable,SyncWithBaseTables,UnsyncTables,CommittedVersion,RowCount
1790240559693,p2026091410,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 10:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 11:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559714,p2026091411,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 11:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 12:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559735,p2026091412,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 12:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 13:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559756,p2026091413,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 13:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 14:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559777,p2026091414,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 14:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 15:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559798,p2026091415,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 15:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 16:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559819,p2026091416,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 16:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 17:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559840,p2026091417,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 17:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 18:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559861,p2026091418,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 18:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 19:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240559882,p2026091419,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 19:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-14 20:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1


,PartitionId,PartitionName,VisibleVersion,VisibleVersionTime,State,PartitionKey,Range,DistributionKey,Buckets,ReplicationNum,...,RemoteStoragePolicy,LastConsistencyCheckTime,DataSize,IsInMemory,ReplicaAllocation,IsMutable,SyncWithBaseTables,UnsyncTables,CommittedVersion,RowCount
0,1790240559693,p2026091410,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 10:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
1,1790240559714,p2026091411,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 11:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
2,1790240559735,p2026091412,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 12:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
3,1790240559756,p2026091413,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 13:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
4,1790240559777,p2026091414,1,2026-09-24 10:29:17,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-14 14:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,1790240564677,p2026092407,1,2026-09-24 10:29:19,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-24 07:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
238,1790240564698,p2026092408,1,2026-09-24 10:29:19,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-24 08:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
239,1790240564719,p2026092409,1,2026-09-24 10:29:19,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-24 09:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
240,1790240564740,p2026092410,1,2026-09-24 10:29:19,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-24 10:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1


What happens when monthly partitioning is used instead?

In [4]:
lab.execute(r"""
CREATE TABLE `otel_logs_for_month_partition` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200)  NULL,
  `trace_id` varchar(200)  NULL,
  `span_id` text  NULL,
  `severity_number` int  NULL,
  `severity_text` text  NULL,
  `body` text  NULL,
  `resource_attributes` variant  NULL,
  `log_attributes` variant  NULL,
  `scope_name` text  NULL,
  `scope_version` text  NULL
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)()
DISTRIBUTED BY HASH(`service_name`) BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "MONTH",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "12", -- monthly partitions for 12 months
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"enable_single_replica_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Monthly partition table')


0

Use the following command to inspect the partitions. The result contains:

```text
14 rows in set (0.00 sec)
```

This means that one year of history partitions was created, plus the current partition and one pre-created partition, for a total of 14 partitions.

In [5]:
lab.sql(r"""
SHOW PARTITIONS FROM otel_logs_for_month_partition;
""", title='Monthly partitions')


PartitionId,PartitionName,VisibleVersion,VisibleVersionTime,State,PartitionKey,Range,DistributionKey,Buckets,ReplicationNum,StorageMedium,CooldownTime,RemoteStoragePolicy,LastConsistencyCheckTime,DataSize,IsInMemory,ReplicaAllocation,IsMutable,SyncWithBaseTables,UnsyncTables,CommittedVersion,RowCount
1790240564842,p202509,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-09-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2025-10-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564863,p202510,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-10-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2025-11-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564884,p202511,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-11-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2025-12-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564905,p202512,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-12-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-01-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564926,p202601,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-01-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-02-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564947,p202602,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-02-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-03-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564968,p202603,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-03-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-04-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240564989,p202604,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-04-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-05-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565010,p202605,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-05-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-06-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565031,p202606,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-06-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-07-01 00:00:00]; ),service_name,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1


,PartitionId,PartitionName,VisibleVersion,VisibleVersionTime,State,PartitionKey,Range,DistributionKey,Buckets,ReplicationNum,...,RemoteStoragePolicy,LastConsistencyCheckTime,DataSize,IsInMemory,ReplicaAllocation,IsMutable,SyncWithBaseTables,UnsyncTables,CommittedVersion,RowCount
0,1790240564842,p202509,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-09-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
1,1790240564863,p202510,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-10-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
2,1790240564884,p202511,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-11-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
3,1790240564905,p202512,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2025-12-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
4,1790240564926,p202601,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-01-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
5,1790240564947,p202602,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-02-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
6,1790240564968,p202603,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-03-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
7,1790240564989,p202604,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-04-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
8,1790240565010,p202605,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-05-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
9,1790240565031,p202606,1,2026-09-24 10:30:02,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-06-01 00:00:...,service_name,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1


However, monthly partitioning is too coarse-grained, so daily details cannot be analyzed independently. Therefore, in observability scenarios, daily partitioning remains the most reasonable choice unless there is a specific requirement.

## Dynamic partitioning and data governance

Doris supports static partitioning, dynamic partitioning, and automatic partitioning. Tables created by the OTel Doris exporter use dynamic partitioning. What should be considered when using dynamic partitioning in observability scenarios?

Consider another scenario: a very large amount of metric data is collected from the OTel Collector every day, and retaining all of it consumes a large amount of disk space. Metrics are mainly used for monitoring and alerting, and only the most recent month of data is typically required. How can partitioning support this type of data-governance requirement?

In [6]:
lab.execute(r"""
CREATE TABLE `otel_metrics_for_data_governance` (
  `service_name` varchar(200) NULL,
  `timestamp` datetime(6) NULL,
  `service_instance_id` varchar(200) NULL,
  `metric_name` varchar(200) NULL,
  `metric_description` text NULL,
  `metric_unit` text NULL,
  `attributes` variant NULL,
  `start_time` datetime(6) NULL,
  `value` double NULL,
  `exemplars` array<struct<filtered_attributes:map<text,text>,timestamp:datetime(6),value:double,span_id:text,trace_id:text>> NULL,
  `resource_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_metric_name (`metric_name`) USING INVERTED,
  INDEX idx_metric_description (`metric_description`) USING INVERTED,
  INDEX idx_metric_unit (`metric_unit`) USING INVERTED,
  INDEX idx_attributes (`attributes`) USING INVERTED,
  INDEX idx_start_time (`start_time`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`service_name`, `timestamp`)
PARTITION BY RANGE(`timestamp`) ()
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-30",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "40",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"enable_single_replica_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Metrics table with dynamic partitions')


0

Pay particular attention to `"dynamic_partition.start" = "-30"`. It means that Doris retains only the most recent 30 days of data and deletes data older than 30 days.

Also pay attention to `"dynamic_partition.history_partition_num" = "40"` and `"dynamic_partition.create_history_partition" = "true"`:

- `"dynamic_partition.create_history_partition" = "true"` means that historical partitions are created when the table is created.
- `"dynamic_partition.history_partition_num" = "40"` specifies how many historical partitions to create.

In the preceding DDL, `dynamic_partition.history_partition_num` is greater than the absolute value of `dynamic_partition.start`. Although 40 historical partitions are requested, Doris retains only 30. Therefore, only 30 historical partitions exist after the table is created.

Use the following command to inspect the partitions. The result contains:

```text
32 rows in set (0.05 sec)
```

The total of 32 consists of 30 historical partitions, the current partition, and one pre-created partition.

In [7]:
lab.sql(r"""
SHOW PARTITIONS FROM otel_metrics_for_data_governance;
""", title='Dynamic partitions for metrics')


PartitionId,PartitionName,VisibleVersion,VisibleVersionTime,State,PartitionKey,Range,DistributionKey,Buckets,ReplicationNum,StorageMedium,CooldownTime,RemoteStoragePolicy,LastConsistencyCheckTime,DataSize,IsInMemory,ReplicaAllocation,IsMutable,SyncWithBaseTables,UnsyncTables,CommittedVersion,RowCount
1790240565178,p20260825,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-25 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-08-26 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565199,p20260826,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-26 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-08-27 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565220,p20260827,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-27 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-08-28 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565241,p20260828,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-28 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-08-29 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565262,p20260829,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-29 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-08-30 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565283,p20260830,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-30 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-08-31 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565304,p20260831,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-31 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-01 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565325,p20260901,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-01 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-02 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565346,p20260902,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-02 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-03 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1
1790240565367,p20260903,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-03 00:00:00]; ..types: [DATETIMEV2]; keys: [2026-09-04 00:00:00]; ),RANDOM,10,1,HDD,9999-12-31 15:59:59,,NULL,0.000,false,tag.location.default: 1,true,true,NULL,1,-1


,PartitionId,PartitionName,VisibleVersion,VisibleVersionTime,State,PartitionKey,Range,DistributionKey,Buckets,ReplicationNum,...,RemoteStoragePolicy,LastConsistencyCheckTime,DataSize,IsInMemory,ReplicaAllocation,IsMutable,SyncWithBaseTables,UnsyncTables,CommittedVersion,RowCount
0,1790240565178,p20260825,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-25 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
1,1790240565199,p20260826,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-26 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
2,1790240565220,p20260827,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-27 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
3,1790240565241,p20260828,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-28 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
4,1790240565262,p20260829,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-29 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
5,1790240565283,p20260830,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-30 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
6,1790240565304,p20260831,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-08-31 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
7,1790240565325,p20260901,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-01 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
8,1790240565346,p20260902,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-02 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1
9,1790240565367,p20260903,1,2026-09-24 10:30:15,NORMAL,timestamp,[types: [DATETIMEV2]; keys: [2026-09-03 00:00:...,RANDOM,10,1,...,,None,0.000,false,tag.location.default: 1,true,true,None,1,-1


## Automatic partitioning

If the dynamic-partition configuration is too extensive at table-creation time and you do not want to manage pre-created partitions, retained partition counts, or data-governance requirements, automatic partitioning is a suitable choice for observability workloads. Consider the metrics table again:

In [8]:
lab.execute(r"""
CREATE TABLE `otel_metrics_for_auto_partition` (
  `service_name` varchar(200) NOT NULL,
  `timestamp` datetime(6) NOT NULL,
  `service_instance_id` varchar(200) NULL,
  `metric_name` varchar(200) NULL,
  `metric_description` text NULL,
  `metric_unit` text NULL,
  `attributes` variant NULL,
  `start_time` datetime(6) NULL,
  `value` double NULL,
  `exemplars` array<struct<filtered_attributes:map<text,text>,timestamp:datetime(6),value:double,span_id:text,trace_id:text>> NULL,
  `resource_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_metric_name (`metric_name`) USING INVERTED,
  INDEX idx_metric_description (`metric_description`) USING INVERTED,
  INDEX idx_metric_unit (`metric_unit`) USING INVERTED,
  INDEX idx_attributes (`attributes`) USING INVERTED,
  INDEX idx_start_time (`start_time`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`service_name`, `timestamp`)
AUTO PARTITION BY RANGE (date_trunc(`timestamp`, 'day')) ()
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1"
);
""", title='Metrics table with automatic partitions')


0

This DDL is concise. When metrics data is written, Doris automatically creates any partitions that do not exist.

## Bucket distribution strategies

Recall the `otel_logs_for_error` table created when table models for observability scenarios were discussed. The requirement is to analyze log data frequently by day and `service_name`, so `service_name` is often used as a filter in the `WHERE` clause. Its DDL uses HASH bucketing:

In [9]:
lab.sql(r"""
SHOW CREATE TABLE otel_logs_for_error
""", title='ERROR log table layout')


Table,Create Table
otel_logs_for_error,"CREATE TABLE `otel_logs_for_error` ( `timestamp` datetime(6) NULL, `service_name` varchar(200) NULL, `service_instance_id` varchar(200) REPLACE NULL, `trace_id` varchar(200) REPLACE NULL, `span_id` text REPLACE NULL, `severity_number` int REPLACE NULL, `severity_text` text REPLACE NULL, `body` text REPLACE NULL, `resource_attributes` variant REPLACE NULL, `log_attributes` variant REPLACE NULL, `scope_name` text REPLACE NULL, `scope_version` text REPLACE NULL, `count_value` bigint SUM NULL ) ENGINE=OLAP AGGREGATE KEY(`timestamp`, `service_name`) PARTITION BY RANGE(`timestamp`) (PARTITION p20260914 VALUES [('2026-09-14 00:00:00'), ('2026-09-15 00:00:00')), PARTITION p20260915 VALUES [('2026-09-15 00:00:00'), ('2026-09-16 00:00:00')), PARTITION p20260916 VALUES [('2026-09-16 00:00:00'), ('2026-09-17 00:00:00')), PARTITION p20260917 VALUES [('2026-09-17 00:00:00'), ('2026-09-18 00:00:00')), PARTITION p20260918 VALUES [('2026-09-18 00:00:00'), ('2026-09-19 00:00:00')), PARTITION p20260919 VALUES [('2026-09-19 00:00:00'), ('2026-09-20 00:00:00')), PARTITION p20260920 VALUES [('2026-09-20 00:00:00'), ('2026-09-21 00:00:00')), PARTITION p20260921 VALUES [('2026-09-21 00:00:00'), ('2026-09-22 00:00:00')), PARTITION p20260922 VALUES [('2026-09-22 00:00:00'), ('2026-09-23 00:00:00')), PARTITION p20260923 VALUES [('2026-09-23 00:00:00'), ('2026-09-24 00:00:00')), PARTITION p20260924 VALUES [('2026-09-24 00:00:00'), ('2026-09-25 00:00:00')), PARTITION p20260925 VALUES [('2026-09-25 00:00:00'), ('2026-09-26 00:00:00'))) DISTRIBUTED BY HASH(`service_name`) BUCKETS AUTO PROPERTIES ( ""replication_allocation"" = ""tag.location.default: 1"", ""min_load_replica_num"" = ""-1"", ""is_being_synced"" = ""false"", ""dynamic_partition.enable"" = ""true"", ""dynamic_partition.time_unit"" = ""DAY"", ""dynamic_partition.time_zone"" = ""Etc/UTC"", ""dynamic_partition.start"" = ""-10"", ""dynamic_partition.end"" = ""1"", ""dynamic_partition.prefix"" = ""p"", ""dynamic_partition.replication_allocation"" = ""tag.location.default: 1"", ""dynamic_partition.buckets"" = ""10"", ""dynamic_partition.create_history_partition"" = ""true"", ""dynamic_partition.history_partition_num"" = ""10"", ""dynamic_partition.hot_partition_num"" = ""0"", ""dynamic_partition.reserved_history_periods"" = ""NULL"", ""dynamic_partition.storage_policy"" = """", ""storage_medium"" = ""hdd"", ""storage_format"" = ""V2"", ""inverted_index_storage_format"" = ""V2"", ""light_schema_change"" = ""true"", ""compaction_policy"" = ""time_series"", ""time_series_compaction_goal_size_mbytes"" = ""1024"", ""time_series_compaction_file_count_threshold"" = ""2000"", ""time_series_compaction_time_threshold_seconds"" = ""3600"", ""time_series_compaction_empty_rowsets_threshold"" = ""5"", ""time_series_compaction_level_threshold"" = ""1"", ""disable_auto_compaction"" = ""false"", ""enable_single_replica_compaction"" = ""false"", ""group_commit_interval_ms"" = ""10000"", ""group_commit_data_bytes"" = ""134217728"" );"


,Table,Create Table
0,otel_logs_for_error,CREATE TABLE `otel_logs_for_error` (\n `times...


This bucketing strategy places `service_name` values with the same hash value into the same bucket. When the `WHERE` clause contains `service_name`, the query can target a specific bucket and improve query performance.

However, the selected bucket column is not always perfectly distributed. Consider the same requirement, where `service_name` is queried frequently, but its values are affected by data skew. Suppose `service_name` has only two values, `mobile` and `PC`, with 1 billion rows for `mobile` and 1 million rows for `PC`. All 1 billion `mobile` rows are placed in the same bucket, creating a data hotspot that can impose significant performance overhead during data loading and queries.

For this scenario, use RANDOM bucketing, as shown by the logs table already created by the OTel Doris exporter:

In [10]:
lab.sql(r"""
SHOW CREATE TABLE otel_logs
""", title='OTel log table layout')


Table,Create Table
otel_logs,"CREATE TABLE `otel_logs` ( `timestamp` datetime(6) NULL, `service_name` varchar(200) NULL, `service_instance_id` varchar(200) NULL, `trace_id` varchar(200) NULL, `span_id` text NULL, `severity_number` int NULL, `severity_text` text NULL, `body` text NULL, `resource_attributes` variant NULL, `log_attributes` variant NULL, `scope_name` text NULL, `scope_version` text NULL, INDEX idx_service_name (`service_name`) USING INVERTED, INDEX idx_timestamp (`timestamp`) USING INVERTED, INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED, INDEX idx_trace_id (`trace_id`) USING INVERTED, INDEX idx_span_id (`span_id`) USING INVERTED, INDEX idx_severity_number (`severity_number`) USING INVERTED, INDEX idx_body (`body`) USING INVERTED PROPERTIES(""lower_case"" = ""true"", ""parser"" = ""unicode"", ""support_phrase"" = ""true""), INDEX idx_severity_text (`severity_text`) USING INVERTED, INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED, INDEX idx_log_attributes (`log_attributes`) USING INVERTED, INDEX idx_scope_name (`scope_name`) USING INVERTED, INDEX idx_scope_version (`scope_version`) USING INVERTED ) ENGINE=OLAP DUPLICATE KEY(`timestamp`, `service_name`) PARTITION BY RANGE(`timestamp`) (PARTITION p20260921 VALUES [('2026-09-21 00:00:00'), ('2026-09-22 00:00:00')), PARTITION p20260922 VALUES [('2026-09-22 00:00:00'), ('2026-09-23 00:00:00')), PARTITION p20260923 VALUES [('2026-09-23 00:00:00'), ('2026-09-24 00:00:00')), PARTITION p20260924 VALUES [('2026-09-24 00:00:00'), ('2026-09-25 00:00:00')), PARTITION p20260925 VALUES [('2026-09-25 00:00:00'), ('2026-09-26 00:00:00'))) DISTRIBUTED BY RANDOM BUCKETS AUTO PROPERTIES ( ""replication_allocation"" = ""tag.location.default: 1"", ""min_load_replica_num"" = ""-1"", ""is_being_synced"" = ""false"", ""dynamic_partition.enable"" = ""true"", ""dynamic_partition.time_unit"" = ""DAY"", ""dynamic_partition.time_zone"" = ""Etc/UTC"", ""dynamic_partition.start"" = ""-7"", ""dynamic_partition.end"" = ""1"", ""dynamic_partition.prefix"" = ""p"", ""dynamic_partition.replication_allocation"" = ""tag.location.default: 1"", ""dynamic_partition.buckets"" = ""10"", ""dynamic_partition.create_history_partition"" = ""true"", ""dynamic_partition.history_partition_num"" = ""3"", ""dynamic_partition.hot_partition_num"" = ""0"", ""dynamic_partition.reserved_history_periods"" = ""NULL"", ""dynamic_partition.storage_policy"" = """", ""storage_medium"" = ""hdd"", ""storage_format"" = ""V2"", ""inverted_index_storage_format"" = ""V2"", ""light_schema_change"" = ""true"", ""compaction_policy"" = ""time_series"", ""time_series_compaction_goal_size_mbytes"" = ""1024"", ""time_series_compaction_file_count_threshold"" = ""2000"", ""time_series_compaction_time_threshold_seconds"" = ""3600"", ""time_series_compaction_empty_rowsets_threshold"" = ""5"", ""time_series_compaction_level_threshold"" = ""1"", ""disable_auto_compaction"" = ""false"", ""enable_single_replica_compaction"" = ""false"", ""group_commit_interval_ms"" = ""10000"", ""group_commit_data_bytes"" = ""134217728"" );"


,Table,Create Table
0,otel_logs,CREATE TABLE `otel_logs` (\n `timestamp` date...


If hotspot data is present, RANDOM bucketing can significantly improve load and query performance.

## Bucket count

The bucket count also requires careful consideration when configuring buckets. The bucket count specifies how many buckets exist under one partition and can be configured manually. In the following DDL, `DISTRIBUTED BY RANDOM BUCKETS 10` means that RANDOM bucketing is used and the bucket count is 10:

In [11]:
lab.execute(r"""
CREATE TABLE `otel_metrics_for_bucket_number` (
  `service_name` varchar(200) NOT NULL,
  `timestamp` datetime(6) NOT NULL,
  `service_instance_id` varchar(200) NULL,
  `metric_name` varchar(200) NULL,
  `metric_description` text NULL,
  `metric_unit` text NULL,
  `attributes` variant NULL,
  `start_time` datetime(6) NULL,
  `value` double NULL,
  `exemplars` array<struct<filtered_attributes:map<text,text>,timestamp:datetime(6),value:double,span_id:text,trace_id:text>> NULL,
  `resource_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_metric_name (`metric_name`) USING INVERTED,
  INDEX idx_metric_description (`metric_description`) USING INVERTED,
  INDEX idx_metric_unit (`metric_unit`) USING INVERTED,
  INDEX idx_attributes (`attributes`) USING INVERTED,
  INDEX idx_start_time (`start_time`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`service_name`, `timestamp`)
AUTO PARTITION BY RANGE (date_trunc(`timestamp`, 'day')) ()
DISTRIBUTED BY RANDOM BUCKETS 10
PROPERTIES (
"replication_allocation" = "tag.location.default: 1"
);
""", title='Explicit bucket-count table')


0

If the bucket count can be configured manually, what difference does the count make? For the metrics table, suppose only one bucket is configured. All data in a partition is then stored in that single bucket. When metrics data is queried, only partitions can be pruned, and bucket-level pruning is unavailable, resulting in very low query efficiency. If too many buckets are configured, metadata management becomes difficult and places substantial pressure on the FE.

What bucket count is reasonable in observability scenarios? You can estimate it from the amount of data written each day. Suppose the logs data reaches 100 GB per day before compression and daily partitioning is used, so one partition contains approximately 100 GB. After compression, the size is approximately 30 GB. If one bucket is expected to store 1 to 10 GB after compression, the table should be configured with approximately 3 to 30 buckets.

This calculation is inconvenient because the daily data volume can vary, making manual bucket configuration difficult. Is there a simpler bucket-count strategy? The OTel Doris exporter already demonstrates one:

In [12]:
lab.sql(r"""
SHOW CREATE TABLE otel_logs
""", title='OTel log bucket configuration')


Table,Create Table
otel_logs,"CREATE TABLE `otel_logs` ( `timestamp` datetime(6) NULL, `service_name` varchar(200) NULL, `service_instance_id` varchar(200) NULL, `trace_id` varchar(200) NULL, `span_id` text NULL, `severity_number` int NULL, `severity_text` text NULL, `body` text NULL, `resource_attributes` variant NULL, `log_attributes` variant NULL, `scope_name` text NULL, `scope_version` text NULL, INDEX idx_service_name (`service_name`) USING INVERTED, INDEX idx_timestamp (`timestamp`) USING INVERTED, INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED, INDEX idx_trace_id (`trace_id`) USING INVERTED, INDEX idx_span_id (`span_id`) USING INVERTED, INDEX idx_severity_number (`severity_number`) USING INVERTED, INDEX idx_body (`body`) USING INVERTED PROPERTIES(""lower_case"" = ""true"", ""parser"" = ""unicode"", ""support_phrase"" = ""true""), INDEX idx_severity_text (`severity_text`) USING INVERTED, INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED, INDEX idx_log_attributes (`log_attributes`) USING INVERTED, INDEX idx_scope_name (`scope_name`) USING INVERTED, INDEX idx_scope_version (`scope_version`) USING INVERTED ) ENGINE=OLAP DUPLICATE KEY(`timestamp`, `service_name`) PARTITION BY RANGE(`timestamp`) (PARTITION p20260921 VALUES [('2026-09-21 00:00:00'), ('2026-09-22 00:00:00')), PARTITION p20260922 VALUES [('2026-09-22 00:00:00'), ('2026-09-23 00:00:00')), PARTITION p20260923 VALUES [('2026-09-23 00:00:00'), ('2026-09-24 00:00:00')), PARTITION p20260924 VALUES [('2026-09-24 00:00:00'), ('2026-09-25 00:00:00')), PARTITION p20260925 VALUES [('2026-09-25 00:00:00'), ('2026-09-26 00:00:00'))) DISTRIBUTED BY RANDOM BUCKETS AUTO PROPERTIES ( ""replication_allocation"" = ""tag.location.default: 1"", ""min_load_replica_num"" = ""-1"", ""is_being_synced"" = ""false"", ""dynamic_partition.enable"" = ""true"", ""dynamic_partition.time_unit"" = ""DAY"", ""dynamic_partition.time_zone"" = ""Etc/UTC"", ""dynamic_partition.start"" = ""-7"", ""dynamic_partition.end"" = ""1"", ""dynamic_partition.prefix"" = ""p"", ""dynamic_partition.replication_allocation"" = ""tag.location.default: 1"", ""dynamic_partition.buckets"" = ""10"", ""dynamic_partition.create_history_partition"" = ""true"", ""dynamic_partition.history_partition_num"" = ""3"", ""dynamic_partition.hot_partition_num"" = ""0"", ""dynamic_partition.reserved_history_periods"" = ""NULL"", ""dynamic_partition.storage_policy"" = """", ""storage_medium"" = ""hdd"", ""storage_format"" = ""V2"", ""inverted_index_storage_format"" = ""V2"", ""light_schema_change"" = ""true"", ""compaction_policy"" = ""time_series"", ""time_series_compaction_goal_size_mbytes"" = ""1024"", ""time_series_compaction_file_count_threshold"" = ""2000"", ""time_series_compaction_time_threshold_seconds"" = ""3600"", ""time_series_compaction_empty_rowsets_threshold"" = ""5"", ""time_series_compaction_level_threshold"" = ""1"", ""disable_auto_compaction"" = ""false"", ""enable_single_replica_compaction"" = ""false"", ""group_commit_interval_ms"" = ""10000"", ""group_commit_data_bytes"" = ""134217728"" );"


,Table,Create Table
0,otel_logs,CREATE TABLE `otel_logs` (\n `timestamp` date...


`BUCKETS AUTO` enables automatic bucket calculation. It helps Doris determine the bucket count automatically, so you do not need to calculate it manually, making data loading and querying more convenient.

For bucket distribution, HASH bucketing can prune buckets efficiently when the filter matches the bucket column, but it can create hotspots when the column is heavily skewed. RANDOM bucketing is preferable for hotspot data and is used by the tables created by the OTel Doris exporter. When the data volume varies, `BUCKETS AUTO` avoids manual bucket-count calculations.

In summary, daily partitioning is the most suitable approach for most observability scenarios. If only a subset of partitions needs to be retained, dynamic partitioning is a good choice. If ease of use is the priority, automatic partitioning can be used directly.